# Build a Byte-Level BPE Tokenizer

LLM Training Lab, `lltTokenizer`. The cells match the lab page in this app. Steps 1 to 4 also run in the page itself. Steps 5 to 10 need the Hugging Face `tokenizers` library and the data below.

Data: FLORES-200 devtest (CC BY-SA 4.0), Wikipedia 20231101 English, Hindi and Spanish (CC BY-SA 3.0 and GFDL), and the CPython standard library (PSF License). `data.py` records the sources.

In [ ]:
# Runtime check. This lab runs on CPU; it needs no GPU.
import os, shutil, sys, urllib.request
problems = []
if sys.version_info < (3, 11):
    problems.append(f"Python {sys.version.split()[0]} is too old. In Colab: Runtime > Change runtime type, then restart.")
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    if ram < 4:
        problems.append(f"{ram:.1f} GB RAM. The full-size steps need about 2 GB free; close other notebooks.")
except ImportError:
    pass
free = shutil.disk_usage(".").free / 1e9
if free < 1:
    problems.append(f"{free:.1f} GB free disk. The data needs about 60 MB plus 1 GB of headroom.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
except Exception as e:
    problems.append(f"No internet ({type(e).__name__}). On Kaggle: Settings > Internet on (needs a verified phone).")
print("Python", sys.version.split()[0])
print("\n".join(problems) if problems else "Runtime OK. No GPU needed for this lab.")

In [ ]:
!pip install -q tokenizers==0.23.3 huggingface_hub==2.4.0 datasets==5.1.0 psutil

## Setup. Write `bpe.py` (the tokenizer from steps 2 to 4, 160 lines) and `data.py`

In [ ]:
%%writefile bpe.py
"""Byte-level BPE tokenizer in pure Python (LLM Training Lab, lltTokenizer).

The same functions the lab page builds step by step, collected in one file so the
notebook and measure.py import one copy. No dependencies beyond the standard library,
so it also runs in Pyodide.

Pre-tokenization follows GPT-2's rules without the `regex` module:
  's 't 're 've 'm 'll 'd | optional space + letters | optional space + numbers |
  optional space + other symbols | whitespace not followed by a non-space | whitespace
Letters are str.isalpha() (Unicode category L), numbers str.isnumeric(), whitespace
str.isspace(). measure.py checks the result against Hugging Face's GPT-2 tokenizer.
"""
import unicodedata
from collections import Counter
from itertools import groupby

CONTRACTIONS = ("s", "t", "re", "ve", "m", "ll", "d")


def kind(ch, marks=False):
    if ch.isalpha() or (marks and unicodedata.category(ch)[0] == "M"):
        return "L"
    if ch.isnumeric():
        return "N"
    if ch.isspace():
        return "S"
    return "P"


def pretokenize(text, digits="runs", marks=False):
    """Split text into pieces. digits='runs' keeps digit runs whole (GPT-2);
    digits='single' splits every digit into its own piece. marks=True counts combining
    marks (Devanagari vowel signs and virama, category M) as letters, so a Hindi word
    stays one piece; GPT-2's rule (marks=False) cuts it at every vowel sign."""
    key = (lambda ch: kind(ch, True)) if marks else kind
    runs = [[k, "".join(g)] for k, g in groupby(text, key)]
    pieces, lead = [], ""
    for i, (k, run) in enumerate(runs):
        if not run:
            continue
        last = i == len(runs) - 1
        if k == "S":
            if last:
                pieces.append(run)
                continue
            if len(run) > 1:
                pieces.append(run[:-1])
            if run[-1] == " ":
                lead = " "
            else:
                pieces.append(run[-1])
            continue
        if run == "'" and not lead and not last and runs[i + 1][0] == "L":
            nxt = runs[i + 1][1]
            c = next((c for c in CONTRACTIONS if nxt.startswith(c)), None)
            if c:
                pieces.append("'" + c)
                runs[i + 1][1] = nxt[len(c):]
                continue
        if k == "N" and digits == "single":
            pieces.append(lead + run[0])
            pieces.extend(run[1:])
        else:
            pieces.append(lead + run)
        lead = ""
    return pieces


def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train(text, vocab_size, use_pretokenize=True, digits="runs", verbose=False):
    """Learn merges. Returns {pair: new_id}; new_id order is the merge rank."""
    pieces = pretokenize(text, digits) if use_pretokenize else [text]
    words = Counter(tuple(p.encode("utf-8")) for p in pieces)
    merges = {}
    for new_id in range(256, vocab_size):
        stats = Counter()
        for w, n in words.items():
            for pair in zip(w, w[1:]):
                stats[pair] += n
        if not stats:
            break
        pair = max(stats, key=stats.get)
        merges[pair] = new_id
        words = {tuple(merge(list(w), pair, new_id)): n for w, n in words.items()}
        if verbose and new_id % 500 == 0:
            print(new_id, stats[pair])
    return merges


def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), i in merges.items():
        vocab[i] = vocab[a] + vocab[b]
    return vocab


def encode_piece(b, merges):
    ids = list(b)
    while len(ids) >= 2:
        pair = min(zip(ids, ids[1:]), key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids


def encode(text, merges, digits="runs", cache=None):
    out = []
    for p in pretokenize(text, digits):
        if cache is not None:
            ids = cache.get(p)
            if ids is None:
                ids = cache[p] = encode_piece(p.encode("utf-8"), merges)
        else:
            ids = encode_piece(p.encode("utf-8"), merges)
        out.extend(ids)
    return out


def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8")


def bytes_to_unicode():
    """GPT-2's map from bytes to printable characters, used in its merges file."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


def merges_from_hf(merge_list):
    """Turn a Hugging Face merges list (pairs of byte-mapped strings, rank order)
    into this file's {pair: new_id} form. Returns (merges, vocab)."""
    byte_of = {c: b for b, c in bytes_to_unicode().items()}
    to_bytes = lambda s: bytes(byte_of[c] for c in s)
    id_of = {bytes([i]): i for i in range(256)}
    merges = {}
    for r, (a, b) in enumerate(merge_list):
        ba, bb = to_bytes(a), to_bytes(b)
        new_id = 256 + r
        merges[(id_of[ba], id_of[bb])] = new_id
        id_of[ba + bb] = new_id
    return merges, build_vocab(merges)

In [ ]:
%%writefile data.py
"""Build the text used by the lltTokenizer lab. Every source is licence-clear and recorded.

  FLORES-200 dev and devtest (English, Hindi, Spanish), CC BY-SA 4.0
      https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz
      The same sentences in every language, so tokens per word compares like with like.
  Wikipedia 20231101 (English, Hindi, Spanish), CC BY-SA 3.0 and GFDL
      Hugging Face dataset wikimedia/wikipedia, streamed, first articles in file order.
  Python source code: the CPython standard library of the running interpreter, PSF licence.

Usage: python data.py   (writes to ~/.cache/lltlab/tokenizer, about 30 MB)
"""
import hashlib
import json
import os
import sys
import sysconfig
import tarfile
import urllib.request
from pathlib import Path

CACHE = Path(os.environ.get("LLT_TOK_CACHE", Path.home() / ".cache" / "lltlab" / "tokenizer"))
FLORES_URL = "https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz"
FLORES_SHA256 = "b8b0b76783024b85797e5cc75064eb83fc5288b41e9654dabc7be6ae944011f6"
LANGS = {"en": "eng_Latn", "hi": "hin_Deva", "es": "spa_Latn"}
WIKI_BYTES = 8_000_000      # per language, training text
CODE_TRAIN_BYTES = 4_000_000
CODE_HELDOUT_BYTES = 150_000


def flores():
    out = CACHE / "flores"
    if all((out / f"{c}.{s}").exists() for c in LANGS for s in ("dev", "devtest")):
        return out
    out.mkdir(parents=True, exist_ok=True)
    tgz = CACHE / "flores200_dataset.tar.gz"
    if not tgz.exists():
        urllib.request.urlretrieve(FLORES_URL, tgz)
    digest = hashlib.sha256(tgz.read_bytes()).hexdigest()
    assert digest == FLORES_SHA256, f"FLORES archive changed: {digest}"
    with tarfile.open(tgz) as tf:
        for code, name in LANGS.items():
            for split in ("dev", "devtest"):
                member = tf.getmember(f"./flores200_dataset/{split}/{name}.{split}")
                (out / f"{code}.{split}").write_bytes(tf.extractfile(member).read())
    return out


def wikipedia():
    from datasets import load_dataset
    meta = {}
    for code in LANGS:
        path = CACHE / f"wiki_{code}.txt"
        mpath = CACHE / f"wiki_{code}.json"
        if path.exists() and mpath.exists():
            meta[code] = json.loads(mpath.read_text())
            continue
        ds = load_dataset("wikimedia/wikipedia", f"20231101.{code}", split="train", streaming=True)
        parts, size, n, first = [], 0, 0, None
        for ex in ds:
            first = first or ex["url"]
            parts.append(ex["text"])
            size += len(ex["text"].encode("utf-8"))
            n += 1
            if size >= WIKI_BYTES:
                break
        path.write_bytes("\n\n".join(parts).encode("utf-8"))
        meta[code] = {"articles": n, "bytes": size, "firstUrl": first, "lastUrl": ex["url"]}
        mpath.write_text(json.dumps(meta[code]))
    return meta


def python_code():
    lib = Path(sysconfig.get_paths()["stdlib"])
    skip = ("site-packages", "test", "tests", "idlelib", "lib2to3", "turtledemo")
    files = sorted(p for p in lib.rglob("*.py") if not any(s in p.relative_to(lib).parts for s in skip))
    train, held = [], []
    tsize = hsize = 0
    for i, p in enumerate(files):
        text = p.read_text(encoding="utf-8", errors="strict") if p.stat().st_size < 400_000 else ""
        if not text:
            continue
        rel = str(p.relative_to(lib)).replace(os.sep, "/")
        if i % 10 == 0 and hsize < CODE_HELDOUT_BYTES:
            held.append((rel, text)); hsize += len(text.encode("utf-8"))
        elif i % 10 != 0 and tsize < CODE_TRAIN_BYTES:
            train.append((rel, text)); tsize += len(text.encode("utf-8"))
    # write_bytes, not write_text: on Windows write_text turns every newline into CRLF.
    (CACHE / "code_train.txt").write_bytes("\n\n".join(t for _, t in train).encode("utf-8"))
    (CACHE / "code_heldout.txt").write_bytes("\n\n".join(t for _, t in held).encode("utf-8"))
    meta = {"python": sys.version.split()[0], "trainFiles": len(train), "trainBytes": tsize,
            "heldoutFiles": [r for r, _ in held], "heldoutBytes": hsize}
    (CACHE / "code.json").write_text(json.dumps(meta, indent=1))
    return meta


def build():
    CACHE.mkdir(parents=True, exist_ok=True)
    flores()
    wiki = wikipedia()
    code = python_code() if not (CACHE / "code.json").exists() else json.loads((CACHE / "code.json").read_text())
    return {"wikipedia": wiki, "code": code}


def _read(name):
    path = CACHE / name
    if not path.exists():
        build()
    return path.read_bytes().decode("utf-8")


# Helpers the notebook cells and measure.py share, so both read the same text the same way.
def heldout():
    """FLORES-200 devtest per language ('en', 'hi', 'es') and held-out Python code ('code')."""
    out = {code: _read(f"flores/{code}.devtest") for code in LANGS}
    out["code"] = _read("code_heldout.txt")
    return out


def train_texts():
    """Training text per source: Wikipedia 'en', 'hi', 'es' and Python 'code'."""
    out = {code: _read(f"wiki_{code}.txt") for code in LANGS}
    out["code"] = _read("code_train.txt")
    return out


def train_slice(n_bytes=250_000):
    """The first n_bytes of each training source, joined: 1 MB for the default."""
    parts = [t.encode("utf-8")[:n_bytes].decode("utf-8", "ignore") for t in train_texts().values()]
    return "\n".join(parts)


def lines(texts):
    """Every line of every text, the unit the trainers and the counts use."""
    for text in texts:
        yield from text.splitlines()


def heldout_texts():
    """The 3,682 texts of the parity check: FLORES lines, then code chunks (files split on blank lines)."""
    ho = heldout()
    texts = [line for code in LANGS for line in ho[code].splitlines() if line]
    return texts + [c for c in ho["code"].split("\n\n") if c]


if __name__ == "__main__":
    CACHE.mkdir(parents=True, exist_ok=True)
    print("FLORES ->", flores())
    print("Wikipedia ->", wikipedia())
    m = python_code()
    print("Code ->", m["trainFiles"], "train files,", len(m["heldoutFiles"]), "held-out files")

## Step 1. Start from UTF-8 bytes

In [ ]:
samples = {
    "English": "Tokenizers turn text into numbers.",
    "Hindi": "टोकनाइज़र टेक्स्ट को संख्याओं में बदलता है।",
    "Spanish": "Los tokenizadores convierten el texto en números.",
    "Python": "def add(a, b): return a + b",
}
for name, text in samples.items():
    data = text.encode("utf-8")
    print(f"{name:8} {len(text):3} characters {len(data):3} bytes "
          f"{len(data) / len(text):.2f} bytes per character")

print(list("n".encode("utf-8")), list("न".encode("utf-8")), list("ñ".encode("utf-8")))
ids = list(samples["Hindi"].encode("utf-8"))
print(bytes(ids).decode("utf-8") == samples["Hindi"])

## Step 2. Count byte pairs and merge the most frequent one

In [ ]:
import time
from collections import Counter

text = """The tokenizer reads text as bytes. It counts every pair of neighbouring bytes and merges the most frequent pair into a new token. Then it counts again. After a few hundred merges, common words such as the, and, token and text are single tokens.
टोकनाइज़र टेक्स्ट को बाइट्स के रूप में पढ़ता है। वह पास-पास के हर जोड़े को गिनता है और सबसे ज़्यादा आने वाले जोड़े को एक नए टोकन में जोड़ देता है। फिर वह दोबारा गिनता है। कुछ सौ मर्ज के बाद आम शब्द एक-एक टोकन बन जाते हैं।
El tokenizador lee el texto como bytes. Cuenta cada par de bytes vecinos y une el par más frecuente en un token nuevo. Después vuelve a contar. Tras unos cientos de fusiones, las palabras comunes como el, de, token y texto son un solo token."""

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

def train(ids, num_merges):
    merges = {}
    for new_id in range(256, 256 + num_merges):
        pairs = Counter(zip(ids, ids[1:]))
        best = max(pairs, key=pairs.get)
        merges[best] = new_id
        ids = merge(ids, best, new_id)
    return merges, ids

start = time.perf_counter()
data = list(text.encode("utf-8"))
merges, ids = train(data, 100)
print(f"{len(data)} bytes -> {len(ids)} tokens in {time.perf_counter() - start:.2f} s")

vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]
print([vocab[i].decode("utf-8", "replace") for i in range(256, 266)])
crossing = [vocab[i] for i in merges.values() if b" " in vocab[i].lstrip(b" ")]
print(len(crossing), "of 100 new tokens hold a space after a non-space byte")
print([t.decode("utf-8", "replace") for t in crossing if t.isascii()][:6])

## Step 3. Split text into pieces before merging

In [ ]:
from collections import Counter
from itertools import groupby

def kind(ch):
    return "L" if ch.isalpha() else "N" if ch.isnumeric() else "S" if ch.isspace() else "P"

def pretokenize(text):
    runs = [[k, "".join(g)] for k, g in groupby(text, kind)]
    pieces, lead = [], ""
    for i, (k, run) in enumerate(runs):
        last = i == len(runs) - 1
        if not run:
            continue
        if k == "S" and not last:
            if len(run) > 1:
                pieces.append(run[:-1])
            if run[-1] == " ":
                lead = " "
            else:
                pieces.append(run[-1])
            continue
        if run == "'" and not lead and not last and runs[i + 1][0] == "L":
            nxt = runs[i + 1][1]
            c = next((c for c in ("s", "t", "re", "ve", "m", "ll", "d") if nxt.startswith(c)), "")
            if c:
                pieces.append("'" + c)
                runs[i + 1][1] = nxt[len(c):]
                continue
        pieces.append(lead + run)
        lead = ""
    return pieces

print(pretokenize("It's 2026, and x  = 10;"))
print(pretokenize("टोकन बनता है।"))
text = "The tokenizer reads text as bytes. El tokenizador lee el texto como bytes."
pieces = pretokenize(text)
print("".join(pieces) == text, len(pieces), "pieces")

raw = Counter(zip(text, text[1:]))
inside = Counter(pair for p in pieces for pair in zip(p, p[1:]))
ends_word = lambda pairs: sum(n for (a, b), n in pairs.items() if b == " " and a != " ")
print("pairs that end a word with a space: raw text", ends_word(raw), "| inside pieces", ends_word(inside))

## Step 4. Encode by merge rank, decode, and test the round trip

In [ ]:
import random
from collections import Counter

text = "The tokenizer reads text as bytes. टोकनाइज़र टेक्स्ट को बाइट्स के रूप में पढ़ता है। El tokenizador lee el texto como bytes."

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

ids, merges = list(text.encode("utf-8")), {}
for new_id in range(256, 256 + 40):  # step 2's trainer
    pairs = Counter(zip(ids, ids[1:]))
    best = max(pairs, key=pairs.get)
    merges[best] = new_id
    ids = merge(ids, best, new_id)
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]

def encode(s):
    ids = list(s.encode("utf-8"))
    while len(ids) > 1:
        pair = min(zip(ids, ids[1:]), key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8")

print(encode("the text"), decode(encode("the text")))
rng = random.Random(0)
failures = 0
for _ in range(1000):
    chars = [rng.choice([rng.randrange(32, 127), rng.randrange(0x900, 0x980), rng.randrange(0xE000, 0x110000)])
             for _ in range(rng.randrange(0, 30))]
    s = "".join(map(chr, chars))
    failures += decode(encode(s)) != s
print("round-trip failures in 1000 random strings:", failures)

## Download the data (about 60 MB, a minute or two)

In [ ]:
import data
print(data.build())

## Step 5. Check your encoder against GPT-2's tokenizer

In [ ]:
import json
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer
import bpe, data

path = hf_hub_download("gpt2", "tokenizer.json")
spec = json.load(open(path, encoding="utf-8"))
pairs = [m.split(" ") if isinstance(m, str) else m for m in spec["model"]["merges"]]
merges, vocab = bpe.merges_from_hf(pairs)
gpt2 = Tokenizer.from_file(path)
byte_of = {c: b for b, c in bpe.bytes_to_unicode().items()}

def library_tokens(text):
    return [bytes(byte_of[c] for c in t) for t in gpt2.encode(text).tokens]

texts = data.heldout_texts()
same = sum([vocab[i] for i in bpe.encode(t, merges)] == library_tokens(t) for t in texts)
print(len(merges), "merges loaded;", same, "of", len(texts), "texts give the same tokens")

## Step 6. Train the same tokenizer with Hugging Face tokenizers

In [ ]:
import time
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
import bpe, data

def train_bpe(vocab_size, texts, pre=None):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre or pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

text = data.train_slice(250_000)  # 250 KB of each source, 1 MB in all
start = time.perf_counter()
ours = bpe.train(text, 1024)
t_ours = time.perf_counter() - start
start = time.perf_counter()
lib = train_bpe(1024, text.splitlines())
t_lib = time.perf_counter() - start
print(f"train to 1,024 tokens: bpe.py {t_ours:.1f} s, tokenizers {t_lib:.2f} s")

held = data.heldout()
for lang in ("en", "hi"):
    cache = {}
    n_ours = sum(len(bpe.encode(x, ours, cache=cache)) for x in held[lang].splitlines())
    n_lib = sum(len(e.ids) for e in lib.encode_batch(held[lang].splitlines()))
    size = len(held[lang].encode("utf-8"))
    print(f"{lang} bytes per token: bpe.py {size / n_ours:.2f}, tokenizers {size / n_lib:.2f}")

gpt2 = Tokenizer.from_file(hf_hub_download("gpt2", "tokenizer.json"))
lines = [x for lang in ("en", "hi", "es") for x in held[lang].splitlines()]
start = time.perf_counter()
cache = {}
n = sum(len(bpe.encode(x, merges, cache=cache)) for x in lines)  # GPT-2's merges, step 5
t_ours = time.perf_counter() - start
start = time.perf_counter()
n_lib = sum(len(e.ids) for e in gpt2.encode_batch(lines))
t_lib = time.perf_counter() - start
print(f"encode {n:,} tokens: bpe.py {t_ours:.2f} s, tokenizers {t_lib:.2f} s, same count {n == n_lib}")

## Step 7. Sweep the vocabulary size

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer
import data

train = data.train_texts()
held = data.heldout()

def bytes_per_token(tok, text):
    n = sum(len(e.ids) for e in tok.encode_batch(text.splitlines()))
    return round(len(text.encode("utf-8")) / n, 2)

gpt2 = Tokenizer.from_file(hf_hub_download("gpt2", "tokenizer.json"))
print("vocab    en    hi    es    code")
print(" gpt2", [bytes_per_token(gpt2, held[k]) for k in ("en", "hi", "es", "code")])
for size in (512, 1024, 2048, 4096, 8192, 16384, 32768, 65536):
    tok = train_bpe(size, data.lines(train.values()))  # train_bpe from step 6
    print(f"{size:5}", [bytes_per_token(tok, held[k]) for k in ("en", "hi", "es", "code")])
    if size == 32768:
        ours = tok

## Step 8. Turn pre-tokenization off

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, pre_tokenizers

no_regex = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
on = train_bpe(8192, data.lines(train.values()))
off = train_bpe(8192, data.lines(train.values()), pre=no_regex)
gpt2 = Tokenizer.from_file(hf_hub_download("gpt2", "tokenizer.json"))

def crossing(tok):
    learned = sorted((i, t) for t, i in tok.get_vocab().items() if i >= 256)
    return [t.replace("Ġ", " ") for i, t in learned if "Ġ" in t.lstrip("Ġ")]

for name, tok in (("gpt2", gpt2), ("regex on", on), ("regex off", off)):
    print(f"{name:9} {len(crossing(tok)):5} learned tokens cross a space, "
          f"en {bytes_per_token(tok, held['en'])} bytes per token")
print([t for t in crossing(off) if t.isascii()][:12])

## Step 9. Split numbers into single digits

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, pre_tokenizers

split_digits = pre_tokenizers.Sequence([pre_tokenizers.Digits(individual_digits=True),
                                        pre_tokenizers.ByteLevel(add_prefix_space=False)])
single = train_bpe(32768, data.lines(train.values()), pre=split_digits)
gpt2 = Tokenizer.from_file(hf_hub_download("gpt2", "tokenizer.json"))
numbers = [" " + str(n) for n in range(10000)]

for name, tok in (("gpt2", gpt2), ("digit runs", ours), ("single digits", single)):
    counts = [len(e.ids) for e in tok.encode_batch(numbers)]
    shapes = {tuple(len(t.lstrip("Ġ")) for t in e.tokens) for e in tok.encode_batch(numbers[1000:])}
    print(f"{name:13} {sum(counts) / len(counts):.2f} tokens per number, "
          f"{len(shapes)} ways to split a 4-digit number, "
          f"code {bytes_per_token(tok, held['code'])} bytes per token")

## Step 10. Measure tokens per word across languages

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

repos = {"GPT-2": "gpt2", "SmolLM2": "HuggingFaceTB/SmolLM2-135M", "Qwen3": "Qwen/Qwen3-0.6B"}
toks = {name: Tokenizer.from_file(hf_hub_download(r, "tokenizer.json")) for name, r in repos.items()}
toks["ours 32k"] = ours  # from step 7

print("tokens per word    en    hi    es  code")
for name, tok in toks.items():
    row = []
    for k in ("en", "hi", "es", "code"):
        n = sum(len(e.ids) for e in tok.encode_batch(held[k].splitlines()))
        row.append(round(n / len(held[k].split()), 2))
    print(f"{name:9} {tok.get_vocab_size():7,}", row)

## Compare with our runs

Our recorded runs are in `courses/llm-training/runs/lltTokenizer/`. `measure.py` in the lab folder regenerates every one of them.